# Evaluación del Pipeline RAG con RAGAS
**Proyecto:** ML-Tutor — Tutor de Machine Learning con RAG  
**Documento:** Glosario de ML de Google (697 términos)  
**Framework de evaluación:** RAGAS 0.4.x  

---

## Parámetros del Pipeline

| Parámetro | Valor |
|-----------|-------|
| Documento(s) | Glosario ML de Google — 697 términos en español (`data/glossary.json`) |
| Modelo de embeddings (retrieval) | `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensiones, multilingual) |
| chunk_size / overlap | Sin chunking — cada documento = 1 término completo / sin overlap |
| k (chunks recuperados) | 3 por defecto (modo chat: 4) |
| LLM generador | Ollama `llama3.2` (temperature=0.3) |
| LLM juez (RAGAS) | `gemini-1.5-flash` (Google) vía InstructorLLM |
| Embeddings juez (RAGAS) | `text-embedding-004` (Google) |

## 1. Instalación de dependencias

In [ ]:
# Instalar RAGAS y dependencias necesarias
# Ejecutar solo si no están instaladas
%pip install "ragas>=0.4.0" datasets python-dotenv instructor google-genai langchain-google-genai jsonref --quiet

## 2. Imports y configuración

In [ ]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from rag.chain import MLTutorChain

# RAGAS 0.4.x — métricas singleton (compatibles con evaluate())
from ragas import evaluate, EvaluationDataset, RunConfig
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import InstructorLLM
from ragas.embeddings import LangchainEmbeddingsWrapper

import instructor
from google import genai
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import pandas as pd

print('Imports OK')
print(f'ragas version: {__import__("ragas").__version__}')

## 3. Configurar el LLM juez para RAGAS

RAGAS 0.4.x usa `InstructorLLM` para garantizar salidas JSON estructuradas.

**Antes de correr:** configura tu API key de Google AI Studio:

**Opción A — Variable de entorno:**
```bash
export GOOGLE_API_KEY="AIza..."
```

**Opción B — Archivo `.env` en la raíz del proyecto (está en .gitignore):**
```
GOOGLE_API_KEY=AIza...
```

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

api_key = os.environ.get('GOOGLE_API_KEY')
if not api_key:
    raise EnvironmentError(
        'Falta GOOGLE_API_KEY.\n'
        'Crea un archivo .env en la raíz del proyecto con:\n'
        '  GOOGLE_API_KEY=AIza...'
    )

# gemini-1.5-flash: 1500 req/día en free tier (vs 20/día de gemini-2.5-flash)
gemini_client = genai.Client(api_key=api_key)
judge_llm = InstructorLLM(
    client=instructor.from_genai(gemini_client),
    model='gemini-1.5-flash',
    provider='google',
)

# Embeddings para answer_relevancy — requiere .embed_query() (interfaz LangChain)
judge_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(
        model='models/text-embedding-004',
        google_api_key=api_key,
    )
)

# Asignar juez a los singletons
faithfulness.llm = judge_llm
answer_relevancy.llm = judge_llm
answer_relevancy.embeddings = judge_embeddings
context_precision.llm = judge_llm

METRICS = [faithfulness, answer_relevancy, context_precision]

print('LLM juez: gemini-1.5-flash (Google, vía InstructorLLM)')
print('Embeddings: text-embedding-004 (Google)')
print('LLM generador: llama3.2 (Ollama local)')
print('Métricas configuradas:', [m.name for m in METRICS])

## 4. Verificar Ollama e inicializar el pipeline RAG

> **Antes de correr esta celda:** asegúrate de que Ollama esté corriendo.  
> Si no está activo, abre una terminal y ejecuta `ollama serve`.

In [ ]:
import httpx

OLLAMA_URL = 'http://localhost:11434'

try:
    r = httpx.get(f'{OLLAMA_URL}/api/tags', timeout=3)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f'Ollama corriendo OK. Modelos disponibles: {models}')
    if not any('llama3.2' in m for m in models):
        print('AVISO: llama3.2 no está descargado. Ejecuta:  ollama pull llama3.2')
except Exception as e:
    print('ERROR: Ollama no está corriendo.')
    print('Solución: abre una terminal y ejecuta  ollama serve')
    raise SystemExit('Detener notebook — inicia Ollama primero.') from e

chain = MLTutorChain(model='llama3.2')
print('Pipeline RAG inicializado correctamente.')

## 5. Casos de prueba

| Tipo | Descripción |
|------|-------------|
| **A** | La respuesta está textualmente en el documento |
| **B** | Vocabulario diferente al del documento — prueba los embeddings |
| **C** | Requiere combinar información de varios chunks |
| **D** | El sistema NO debería tener la respuesta — detecta alucinaciones |

In [ ]:
test_cases = [
    # ─── TIPO A: Respuesta textual en el documento ─────────────────────────────
    {
        'tipo': 'A — Literal en documento',
        'question': '¿Qué es el sobreajuste (overfitting) en machine learning?',
        'ground_truth': (
            'El sobreajuste ocurre cuando un modelo aprende demasiado bien '
            'los datos de entrenamiento, incluyendo el ruido y las variaciones '
            'aleatorias, de modo que falla en generalizar a datos nuevos. '
            'Un modelo con overfitting tiene alto rendimiento en entrenamiento '
            'pero bajo rendimiento en la evaluación.'
        ),
    },
    {
        'tipo': 'A — Literal en documento',
        'question': '¿Qué es el descenso de gradiente?',
        'ground_truth': (
            'El descenso de gradiente es una técnica de optimización matemática '
            'usada para entrenar modelos de ML. Consiste en calcular el gradiente '
            'de la función de pérdida con respecto a los parámetros del modelo y '
            'actualizar los parámetros en la dirección opuesta al gradiente para '
            'minimizar el error.'
        ),
    },
    {
        'tipo': 'A — Literal en documento',
        'question': '¿Qué es una función de pérdida (loss function) en machine learning?',
        'ground_truth': (
            'Una función de pérdida mide la diferencia entre la predicción del '
            'modelo y el valor real. El objetivo del entrenamiento es minimizar '
            'esta función. Ejemplos comunes son el error cuadrático medio (MSE) '
            'para regresión y la entropía cruzada para clasificación.'
        ),
    },

    # ─── TIPO B: Vocabulario diferente (prueba embeddings) ─────────────────────
    {
        'tipo': 'B — Vocabulario diferente',
        'question': (
            '¿Cómo se llama el problema donde un algoritmo memoriza '
            'los datos de entrenamiento en lugar de aprender patrones generales?'
        ),
        'ground_truth': (
            'Este problema se llama sobreajuste (overfitting). '
            'Ocurre cuando el modelo se ajusta demasiado a los datos de '
            'entrenamiento, capturando el ruido, y no puede generalizar '
            'correctamente a datos no vistos.'
        ),
    },
    {
        'tipo': 'B — Vocabulario diferente',
        'question': (
            '¿Qué técnica iterativa permite encontrar el mínimo de una función '
            'de costo ajustando los parámetros del modelo paso a paso?'
        ),
        'ground_truth': (
            'La técnica descrita es el descenso de gradiente (gradient descent). '
            'En cada iteración calcula el gradiente de la función de pérdida y '
            'ajusta los pesos del modelo en la dirección que reduce el error, '
            'controlado por la tasa de aprendizaje (learning rate).'
        ),
    },

    # ─── TIPO C: Combinar información de varios chunks ──────────────────────────
    {
        'tipo': 'C — Combinar chunks',
        'question': (
            '¿Cuál es la relación entre el sesgo (bias) y la varianza (variance) '
            'en el rendimiento de un modelo de machine learning?'
        ),
        'ground_truth': (
            'El sesgo y la varianza representan dos fuentes de error en un modelo. '
            'El sesgo alto indica que el modelo es demasiado simple y no captura '
            'los patrones (subajuste). La varianza alta indica que el modelo es '
            'demasiado sensible a los datos de entrenamiento (sobreajuste). '
            'El compromiso sesgo-varianza (bias-variance tradeoff) busca el '
            'equilibrio donde el error total se minimiza.'
        ),
    },
    {
        'tipo': 'C — Combinar chunks',
        'question': (
            '¿En qué se diferencian la regularización L1 y L2 '
            'y cómo ayudan a prevenir el sobreajuste?'
        ),
        'ground_truth': (
            'L1 (Lasso) añade la suma de valores absolutos de los pesos a la '
            'función de pérdida, lo que puede llevar pesos a exactamente cero '
            'produciendo modelos dispersos (sparse). '
            'L2 (Ridge) añade la suma de cuadrados de los pesos, reduciendo su '
            'magnitud pero sin eliminarlos completamente. '
            'Ambas técnicas penalizan pesos grandes y reducen el sobreajuste '
            'al restringir la complejidad del modelo.'
        ),
    },

    # ─── TIPO D: Sistema no debería tener la respuesta (alucinaciones) ──────────
    {
        'tipo': 'D — Fuera del dominio',
        'question': '¿Cuánto costó en dólares desarrollar el modelo GPT-4 de OpenAI?',
        'ground_truth': (
            'Esta información no está disponible en el glosario de ML de Google. '
            'El costo exacto de desarrollo de GPT-4 no ha sido divulgado públicamente '
            'y no es un concepto del glosario de términos de machine learning.'
        ),
    },
    {
        'tipo': 'D — Fuera del dominio',
        'question': (
            '¿Cuál es el nombre completo del CEO de Google DeepMind '
            'y en qué año nació?'
        ),
        'ground_truth': (
            'Esta información biográfica no se encuentra en el glosario de ML '
            'de Google. El documento solo contiene definiciones técnicas de '
            'conceptos de machine learning y ciencia de datos, no información '
            'sobre personas o biografías.'
        ),
    },
    {
        'tipo': 'D — Fuera del dominio',
        'question': '¿Cuál es el precio actual de una suscripción a ChatGPT Plus en Colombia?',
        'ground_truth': (
            'Esta información de precios no se encuentra en el glosario de ML '
            'de Google. El documento solo contiene definiciones técnicas de '
            'términos de machine learning, no precios de servicios comerciales.'
        ),
    },
]

print(f'Total de casos de prueba: {len(test_cases)}')
for i, tc in enumerate(test_cases, 1):
    print(f'  {i:2d}. [{tc["tipo"]}] {tc["question"][:65]}')

## 6. Ejecutar el pipeline RAG para cada pregunta

In [ ]:
def run_rag_pipeline(chain: MLTutorChain, question: str, k: int = 3):
    """Ejecuta el pipeline RAG y retorna respuesta y contextos recuperados."""
    collection = chain._get_collection()
    results = collection.query(
        query_texts=[question],
        n_results=k,
        include=['documents', 'metadatas'],
    )
    contexts = results['documents'][0] if results['documents'] else []

    try:
        answer = chain.chat(question)
    except Exception as e:
        answer = f'[ERROR al generar respuesta: {type(e).__name__}. ¿Está Ollama corriendo?]'

    return answer, contexts


print('Ejecutando pipeline RAG para cada pregunta...')
print('(Cada pregunta tarda ~10-30 seg dependiendo del hardware)\n')

questions     = []
answers       = []
contexts_list = []
ground_truths = []
tipos         = []

for i, tc in enumerate(test_cases, 1):
    print(f'[{i}/{len(test_cases)}] {tc["question"][:65]}')
    answer, contexts = run_rag_pipeline(chain, tc['question'], k=3)

    questions.append(tc['question'])
    answers.append(answer)
    contexts_list.append(contexts)
    ground_truths.append(tc['ground_truth'])
    tipos.append(tc['tipo'])

    print(f'   Contextos: {len(contexts)} chunks recuperados')
    if answer.startswith('[ERROR'):
        print(f'   {answer}')
    else:
        print(f'   Respuesta: {answer[:100]}...')
    print()

errors = sum(1 for a in answers if a.startswith('[ERROR'))
print(f'Pipeline completado. {len(test_cases) - errors}/{len(test_cases)} respuestas generadas exitosamente.')
if errors:
    print(f'AVISO: {errors} pregunta(s) fallaron. Verifica que Ollama esté corriendo con: ollama serve')

## 7. Evaluación con RAGAS

| Métrica | ¿Qué mide? |
|---------|------------|
| **Faithfulness** | ¿Las afirmaciones de la respuesta están soportadas por los contextos recuperados? |
| **Answer Relevancy** | ¿La respuesta es pertinente para la pregunta? |
| **Context Precision** | ¿Los chunks recuperados son relevantes para responder la pregunta? |

In [ ]:
import time, re

# Segundos de espera entre muestras para no saturar el rate limit (15 RPM free tier)
SLEEP_BETWEEN_SAMPLES = 20  # aumentar a 60 si sigue dando 429 por minuto

def evaluate_one(sample, metrics, max_retries=8):
    """Evalúa una sola muestra con reintentos ante rate-limit."""
    ds = EvaluationDataset(samples=[sample])
    rc = RunConfig(timeout=180, max_retries=2, max_wait=10, max_workers=1)

    for attempt in range(max_retries):
        try:
            result = evaluate(dataset=ds, metrics=metrics, run_config=rc, raise_exceptions=True)
            return result.to_pandas().iloc[0]
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                # Extraer el retryDelay que devuelve la API
                m = re.search(r'retryDelay.*?(\d+)s', msg)
                wait = int(m.group(1)) + 5 if m else 65 * (attempt + 1)
                print(f'    ⚠ Rate limit (intento {attempt+1}/{max_retries}). Esperando {wait}s...')
                time.sleep(wait)
            else:
                print(f'    ✗ Error no recuperable: {msg[:120]}')
                break
    print('    ✗ Máximo de reintentos alcanzado — se guardará NaN para esta muestra.')
    return pd.Series({'faithfulness': None, 'answer_relevancy': None, 'context_precision': None})


print('Preparado. Evaluar muestra a muestra con sleep de', SLEEP_BETWEEN_SAMPLES, 'seg entre cada una.')

In [ ]:
samples = [
    SingleTurnSample(
        user_input=q,
        response=a,
        retrieved_contexts=ctx,
        reference=gt,
    )
    for q, a, ctx, gt in zip(questions, answers, contexts_list, ground_truths)
]

print(f'Evaluando {len(samples)} muestras con gemini-2.5-flash como juez...')
print(f'Sleep entre muestras: {SLEEP_BETWEEN_SAMPLES}s  |  Reintentos por rate-limit: 8\n')

rows = []
for i, sample in enumerate(samples):
    print(f'[{i+1}/{len(samples)}] {sample.user_input[:65]}')
    row = evaluate_one(sample, METRICS)
    rows.append(row)

    f  = f"{row.get('faithfulness', float('nan')):.3f}"  if pd.notna(row.get('faithfulness'))  else 'NaN'
    ar = f"{row.get('answer_relevancy', float('nan')):.3f}" if pd.notna(row.get('answer_relevancy')) else 'NaN'
    cp = f"{row.get('context_precision', float('nan')):.3f}" if pd.notna(row.get('context_precision')) else 'NaN'
    print(f'    faithfulness={f}  answer_relevancy={ar}  context_precision={cp}')

    if i < len(samples) - 1:
        print(f'    (esperando {SLEEP_BETWEEN_SAMPLES}s...)')
        time.sleep(SLEEP_BETWEEN_SAMPLES)
    print()

results_df_raw = pd.DataFrame(rows)
print('Evaluación completada.')
print(results_df_raw[['faithfulness', 'answer_relevancy', 'context_precision']])

## 8. Resultados detallados

In [ ]:
df_results = results_df_raw.copy()
df_results.insert(0, 'tipo', tipos)
df_results.insert(1, 'question', questions)

metric_cols = [c for c in ['faithfulness', 'answer_relevancy', 'context_precision'] if c in df_results.columns]

for col in metric_cols:
    df_results[col] = pd.to_numeric(df_results[col], errors='coerce').round(3)

df_display = df_results[['tipo', 'question'] + metric_cols].copy()
df_display['question'] = df_display['question'].str[:55] + '...'

print('=== TABLA DE RESULTADOS ===')
print(df_display.to_string(index=False))

print('\n=== PROMEDIOS POR TIPO ===')
print(df_results.groupby('tipo')[metric_cols].mean().round(3).to_string())

print('\n=== PROMEDIOS GLOBALES ===')
for col in metric_cols:
    val = df_results[col].mean()
    print(f'  {col}: {val:.3f}' if pd.notna(val) else f'  {col}: N/A')

In [ ]:
print('=== TABLA PARA INFORME ===')
print()
header = f"{'Pregunta':<65} {'Faithfulness':>12} {'Ans.Relevancy':>13} {'Ctx.Precision':>14}"
print(header)
print('-' * len(header))

for _, row in df_results.iterrows():
    q      = row['question'][:62] + '...' if len(row['question']) > 62 else row['question']
    f_val  = f"{row['faithfulness']:.3f}"      if 'faithfulness'      in df_results.columns else 'N/A'
    ar_val = f"{row['answer_relevancy']:.3f}"  if 'answer_relevancy'  in df_results.columns else 'N/A'
    cp_val = f"{row['context_precision']:.3f}" if 'context_precision' in df_results.columns else 'N/A'
    print(f"[{row['tipo'][:15]}] {q:<62} {f_val:>12} {ar_val:>13} {cp_val:>14}")

## 9. Exportar resultados

In [ ]:
analisis_template = {
    'A — Literal en documento': (
        'Pregunta directamente mapeada a un término del glosario. '
        'Se espera alta faithfulness y precision si el embedding recupera el término correcto.'
    ),
    'B — Vocabulario diferente': (
        'Prueba la capacidad semántica del modelo de embeddings. '
        'Si la similitud coseno supera el umbral, debe recuperar el término correcto '
        'a pesar del vocabulario paráfraseado.'
    ),
    'C — Combinar chunks': (
        'Requiere que el LLM sintetice información de múltiples términos. '
        'Context precision puede bajar si solo parte de los k=3 chunks son relevantes.'
    ),
    'D — Fuera del dominio': (
        'El glosario no contiene esta información. Se espera baja faithfulness '
        'si el modelo alucina, o alta si reconoce correctamente que no sabe.'
    ),
}

df_results['analisis'] = df_results['tipo'].map(analisis_template).fillna('')

os.makedirs('data', exist_ok=True)

output_path = 'data/ragas_evaluation_results.csv'
df_results.to_csv(output_path, index=False, encoding='utf-8')
print(f'Resultados exportados a: {output_path}')

report_data = []
for i, tc in enumerate(test_cases):
    row = df_results.iloc[i]
    report_data.append({
        'tipo': tc['tipo'],
        'pregunta': tc['question'],
        'ground_truth': tc['ground_truth'],
        'respuesta_generada': answers[i],
        'contextos_recuperados': contexts_list[i],
        'faithfulness':      float(row.get('faithfulness', 0) or 0),
        'answer_relevancy':  float(row.get('answer_relevancy', 0) or 0),
        'context_precision': float(row.get('context_precision', 0) or 0),
    })

json_path = 'data/ragas_evaluation_results.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(report_data, f, ensure_ascii=False, indent=2)
print(f'Datos completos exportados a: {json_path}')

## 10. Análisis Crítico

### Interpretación de métricas

**Faithfulness** mide si cada afirmación en la respuesta puede ser inferida a partir de los contextos recuperados. Un valor bajo indica que el LLM está "alucinando" información que no proviene del glosario.

**Answer Relevancy** mide si la respuesta generada es pertinente para la pregunta formulada. Valores bajos pueden indicar respuestas vagas o que se desvían del tema.

**Context Precision** mide qué tan bien los chunks recuperados son relevantes para la pregunta, usando el ground truth como referencia. Un valor bajo indica que el retriever está devolviendo información poco útil.

---

### Hallazgos esperados por tipo de pregunta

**Tipo A (Literal):** Se espera alta faithfulness y context_precision, ya que el término existe exactamente en el glosario. La respuesta debería estar directamente soportada por los contextos.

**Tipo B (Vocabulario diferente):** La faithfulness depende de si el embedding `paraphrase-multilingual-MiniLM-L12-v2` logra mapear la paráfrasis al término correcto. Si falla el retrieval, la respuesta no tendrá soporte documental. Este tipo de pregunta es el más exigente para el modelo de embeddings.

**Tipo C (Combinar chunks):** Con k=3, el pipeline puede recuperar términos parcialmente relacionados. El LLM debe sintetizar información de múltiples documentos. Es posible observar faithfulness moderada si el modelo extrapola más allá de lo que dicen los contextos.

**Tipo D (Fuera del dominio):** Este es el test más crítico para detectar alucinaciones. Si el modelo responde con información factual no relacionada con el glosario (por ejemplo, inventa el costo de GPT-4), faithfulness será muy baja porque ningún contexto soporta esa afirmación. Un sistema RAG robusto debería responder indicando que la información no está disponible en el documento.

---

### Limitaciones del pipeline

1. **Sin chunking tradicional:** Cada documento es un término completo del glosario. No hay fragmentación por tamaño de tokens, lo que beneficia términos cortos pero podría truncar definiciones largas.

2. **k=3 fijo:** Para preguntas que requieren combinar varios conceptos (Tipo C), tres chunks puede ser insuficiente. El pipeline de chat usa k=4 pero el modo principal usa k=3.

3. **LLM local (llama3.2):** La calidad de la generación está limitada por las capacidades del modelo local. El juez RAGAS usa gpt-4o-mini para mayor objetividad en la evaluación.

4. **Glosario en español con embeddings multilingüe:** El modelo `paraphrase-multilingual-MiniLM-L12-v2` es multilingual pero fue entrenado principalmente en inglés. Preguntas en español con terminología técnica en inglés pueden tener mayor variabilidad semántica.

---

### Conclusión

El pipeline RAG de ML-Tutor es efectivo para preguntas directas (Tipo A y B) donde el modelo de embeddings puede recuperar el contexto correcto. Las preguntas de síntesis (Tipo C) evidencian la limitación de k=3 chunks. El talón de Aquiles del sistema es la detección de preguntas fuera del dominio (Tipo D): sin un mecanismo explícito de rechazo de preguntas no respondibles, el LLM puede fabricar respuestas plausibles pero incorrectas. Se recomienda agregar un paso de verificación de relevancia del contexto antes de generar la respuesta final.